# ANSYS Simulation Toolbox — Overview & Setup

**Who this is for:**  Engineers and students with basic Python knowledge who want to
learn ANSYS MAPDL and AEDT simulation from scratch, understand the underlying
mathematics, and build reusable simulation pipelines.

**What you will learn in this notebook:**
1. What the toolbox does and how it's organized
2. How to install the environment
3. How to verify your ANSYS installation
4. How to check for zombie processes and free ports
5. The config.yaml system — the single source of truth

---

## 1. What This Toolbox Does

This toolbox wraps **ANSYS MAPDL** and **ANSYS HFSS (via PyAEDT)** with:
- A clean Python API that hides the low-level APDL commands
- A Streamlit web app for interactive configuration
- Jupyter notebooks (these!) with step-by-step math and code
- Automatic zombie cleanup and resource management
- A multi-physics pipeline for chaining simulations

### The simulation workflow

```
Config (config.yaml)
       ↓
┌──────────────────────────────────────────────────────────┐
│  1. I/O paths     — where to read geometry / write results │
│  2. Resources     — kill zombies, allocate CPU/RAM         │
│  3. Geometry      — import CDB/STL or build parametrically │
│  4. Boundary cond — Dirichlet, Neumann, periodic, EM BCs   │
│  5. Solver        — Newton-Raphson, substeps, convergence  │
│  6. Diagnostics   — live residuals, port health, RST size  │
│  7. Results       — field plots, probes, VTK, CSV export   │
│  8. Pipeline      — chain stages, sweep parameters         │
└──────────────────────────────────────────────────────────┘
```

---

## 2. Prerequisites

| Software | Version | Source |
|----------|---------|--------|
| Python   | 3.10+   | [python.org](https://python.org) |
| ANSYS Mechanical MAPDL | 2024 R2 or 2025 R2 | ANSYS Customer Portal |
| ANSYS Electronics Desktop (for HFSS) | 2024 R2 | ANSYS Customer Portal |
| nTopology (optional) | 4.x | [ntopology.com](https://ntopology.com) |

In [ ]:
# ── Step 1: Install dependencies ───────────────────────────────────────────
# Run this cell once, then restart the kernel.

# Uncomment and run:
# !pip install ansys-mapdl-core ansys-aedt-core pymapdl
# !pip install pyvista trimesh numpy-stl
# !pip install numpy scipy pandas pyyaml matplotlib seaborn
# !pip install psutil rich streamlit
# !pip install h5py Pillow imageio

print('Installation complete (if you ran the lines above).')

In [ ]:
# ── Step 2: Verify Python environment ─────────────────────────────────────
import sys, importlib

required = [
    'ansys.mapdl.core',
    'numpy', 'scipy', 'pandas', 'matplotlib',
    'yaml', 'psutil', 'pyvista',
]

for pkg in required:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, '__version__', '?')
        print(f'  ✓  {pkg:<30}  {ver}')
    except ImportError:
        print(f'  ✗  {pkg:<30}  NOT INSTALLED')

In [ ]:
# ── Step 3: Verify ANSYS installation ─────────────────────────────────────
import os, pathlib

# Common installation paths (Windows)
ansys_paths = [
    r'D:\ANSYS Inc\v252\ansys\bin\winx64\ansys252.exe',
    r'D:\ANSYS Inc\v251\ansys\bin\winx64\ansys251.exe',
    r'C:\Program Files\ANSYS Inc\v252\ansys\bin\winx64\ansys252.exe',
]

found = [(p, pathlib.Path(p).exists()) for p in ansys_paths]
for path, exists in found:
    icon = '✓' if exists else '✗'
    print(f'  {icon}  {path}')

if not any(e for _, e in found):
    print('\n  ⚠ No ANSYS binary found at expected paths.')
    print('  Update mapdl.custom_bin in config.yaml to point to your ansysXXX.exe')

In [ ]:
# ── Step 4: Check for zombie processes and port availability ───────────────
import sys; sys.path.insert(0, '..')
from ams.resources.manager import kill_ansys_zombies, check_ports

print('=== Port Status ===')
status = check_ports()
for port, occupied in status.items():
    icon = '🔴 OCCUPIED' if occupied else '🟢 free'
    print(f'  Port {port}: {icon}')

print('\n=== Zombie Scan (dry run) ===')
found = kill_ansys_zombies(dry_run=True, verbose=True)

if found:
    print(f'\nFound {len(found)} zombie process(es).')
    print('Run: kill_ansys_zombies(dry_run=False) to kill them.')
else:
    print('\nNo zombies found — system is clean.')

In [ ]:
# ── Step 5: Load the config ─────────────────────────────────────────────────
import yaml
from pathlib import Path

config_path = Path('..') / 'config.yaml'
with open(config_path, encoding='utf-8') as fh:
    cfg = yaml.safe_load(fh)

print('Config loaded successfully.')
print(f'  Problem name : {cfg["problem"]["name"]}')
print(f'  Physics      : {cfg["problem"]["physics"]}')
print(f'  MAPDL port   : {cfg["mapdl"]["port"]}')
print(f'  Output dir   : {cfg["output"]["dir"]}')

print('\nAll top-level config keys:', list(cfg.keys()))

In [ ]:
# ── Step 6: Resource recommendation ──────────────────────────────────────
from ams.resources.manager import ResourceManager

rm   = ResourceManager()
info = rm.system_info()
rec  = rm.recommend(n_elements=50_000)

print(f'System: {info.physical_cpus} cores, {info.total_ram_mb:,} MB RAM')
print(f'Recommendation for 50k elements:')
print(f'  nproc    = {rec.nproc}')
print(f'  ram_mb   = {rec.ram_mb}')
print(f'  port     = {rec.mapdl_port}')
for w in rec.warnings:
    print(f'  ⚠ {w}')

## Summary

You have:
- ✅ Verified the Python environment
- ✅ Located the ANSYS binary
- ✅ Checked for zombie processes and open ports
- ✅ Loaded the config file
- ✅ Gotten a resource recommendation

**Next:** Open `01_io_and_config.ipynb` to learn how the configuration
system works and how to set up your I/O directories.

Or start the Streamlit app from your terminal:
```bash
streamlit run app.py
```